In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("snowflake-to-clickhouse") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.5.4,com.clickhouse:clickhouse-jdbc:0.4.6") \
    .getOrCreate()

pg_url = "jdbc:postgresql://postgres:5432/bigdata_spark"
pg_props = {"user": "user", "password": "password", "driver": "org.postgresql.Driver"}

ch_url = "jdbc:ch://clickhouse:8123/default"
ch_props = {
    "user": "user", 
    "password": "password", 
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "compress": "0",      
    "decompress": "0",
    "create_table_template": "CREATE TABLE {table} ({fields}) ENGINE = Log"
}

In [2]:
tables = ["fact_sales", "dim_products", "dim_customers", "dim_stores", "dim_suppliers", "dim_geography", "dim_pets", "dim_sellers"]

for table in tables:
    df = spark.read.jdbc(url=pg_url, table=table, properties=pg_props)
    df.createOrReplaceTempView(table)

In [3]:
rep1 = spark.sql("""
    SELECT 
        p.name AS product_name,
        p.category AS category_name,
        p.brand,
        SUM(f.quantity) AS total_sold,
        ROUND(SUM(f.total_price), 2) AS total_revenue
    FROM fact_sales f
    JOIN dim_products p ON f.product_id = p.product_id
    GROUP BY p.name, p.category, p.brand
    ORDER BY total_revenue DESC
    LIMIT 10
""")

rep1.write.jdbc(url=ch_url, table="report_product_sales", mode="overwrite", properties=ch_props)

In [4]:
rep2 = spark.sql("""
    SELECT 
        c.customer_id,
        c.first_name,
        c.last_name,
        g.country,
        ROUND(SUM(f.total_price), 2) AS total_spent,
        ROUND(AVG(f.total_price), 2) AS average_check
    FROM fact_sales f
    JOIN dim_customers c ON f.customer_id = c.customer_id
    JOIN dim_geography g ON c.geo_id = g.geo_id
    GROUP BY c.customer_id, c.first_name, c.last_name, g.country
    ORDER BY total_spent DESC
    LIMIT 10
""")

rep2.write.jdbc(url=ch_url, table="report_customer_sales", mode="overwrite", properties=ch_props)

In [5]:
rep3 = spark.sql("""
    SELECT 
        year(f.sale_date) AS sale_year,
        month(f.sale_date) AS sale_month,
        ROUND(SUM(f.total_price), 2) AS monthly_revenue,
        ROUND(AVG(f.total_price), 2) AS avg_order_size,
        SUM(f.quantity) AS items_sold
    FROM fact_sales f
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

rep3.write.jdbc(url=ch_url, table="report_time_sales", mode="overwrite", properties=ch_props)

In [6]:
rep4 = spark.sql("""
    SELECT 
        s.name AS store_name,
        g.city,
        g.country,
        ROUND(SUM(f.total_price), 2) AS store_revenue,
        ROUND(AVG(f.total_price), 2) AS store_avg_check
    FROM fact_sales f
    JOIN dim_stores s ON f.store_id = s.store_id
    JOIN dim_geography g ON s.geo_id = g.geo_id
    GROUP BY 1, 2, 3
    ORDER BY store_revenue DESC
    LIMIT 5
""")

rep4.write.jdbc(url=ch_url, table="report_store_sales", mode="overwrite", properties=ch_props)

In [7]:
rep5 = spark.sql("""
    SELECT 
        sup.name AS supplier_name,
        g.country AS supplier_country,
        ROUND(SUM(f.total_price), 2) AS supplier_revenue,
        ROUND(AVG(p.price), 2) AS avg_item_price
    FROM fact_sales f
    JOIN dim_products p ON f.product_id = p.product_id
    JOIN dim_suppliers sup ON p.supplier_id = sup.supplier_id
    JOIN dim_geography g ON sup.geo_id = g.geo_id
    GROUP BY 1, 2
    ORDER BY supplier_revenue DESC
    LIMIT 5
""")

rep5.write.jdbc(url=ch_url, table="report_supplier_sales", mode="overwrite", properties=ch_props)

In [8]:
raw_df = spark.read.jdbc(url=pg_url, table="raw_data", properties=pg_props)
raw_df.createOrReplaceTempView("raw_data")

rating_df = spark.sql("""
    SELECT product_name AS name, product_brand AS brand, 
           MAX(product_rating) AS rating, MAX(product_reviews) AS reviews 
    FROM raw_data 
    GROUP BY 1, 2
""")
rating_df.createOrReplaceTempView("raw_ratings")

In [9]:
rep6 = spark.sql("""
    SELECT 
        p.name AS product_name,
        r.rating,
        r.reviews,
        SUM(f.quantity) AS sales_volume,
        ROUND(SUM(f.total_price), 2) AS revenue
    FROM fact_sales f
    JOIN dim_products p ON f.product_id = p.product_id
    JOIN raw_ratings r ON p.name = r.name AND p.brand = r.brand
    GROUP BY 1, 2, 3
    ORDER BY r.rating DESC, sales_volume DESC
""")

rep6.write.jdbc(url=ch_url, table="report_quality", mode="overwrite", properties=ch_props)